# Parametric Compilation

Compile a variational circuit once with `FreeParameter`, then reuse it across every iteration of the loop.

**Objectives:**
- Build a `FreeParameter` circuit and run it on the `LocalSimulator` with `inputs={...}` for several angles, seeing one compiled program produce different results
- Read out an exact expectation-value sweep over the free parameter to confirm the circuit is genuinely reusable
- Model the compile-once speedup over a recompile-every-iteration baseline and plot it across a sweep of iteration counts
- Package the same idea as a Braket Hybrid Job entry point, gated so nothing touches AWS

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, FreeParameter, Observable
from braket.devices import LocalSimulator

# Importing the job/AWS modules is free and needs no credentials; we only
# *call* into AWS inside an explicit RUN_ON_AWS guard later in the notebook.
from braket.aws import AwsQuantumJob  # noqa: F401  (imported to show packaging)
from braket.jobs import save_job_result  # noqa: F401
from braket.jobs.metrics import log_metric  # noqa: F401

np.random.seed(7)

# Local simulator: free, instant, no credentials. Everything that runs in this
# notebook runs here. AWS is only ever invoked behind RUN_ON_AWS = False.
device = LocalSimulator()
print("LocalSimulator ready:", device)

## 1. One compiled circuit, many angles

A variational loop submits the *same circuit structure* every iteration, changing only the rotation angles. Without parametric compilation, Braket would compile the program from scratch on every submission — paying the compile cost N times.

`FreeParameter` breaks that. You declare the angle as a free parameter, build the circuit once, and pass concrete values through `inputs={...}` at run time. Braket compiles the program once and substitutes new parameter values on each run.

The circuit below is the canonical example from the GUIDE: an `Rx(theta)` on qubit 0 entangled into qubit 1 with a CNOT.

In [ ]:
theta = FreeParameter("theta")
circ = Circuit().rx(0, theta).cnot(0, 1)

print(circ)
print("\nFree parameters in this circuit:", circ.parameters)

Now run that *same* `circ` object for three different angles. We never rebuild the circuit — we only change `inputs`. The measurement counts shift accordingly:

- `theta = 0`     -> `Rx` is identity, qubit 0 stays |0>, CNOT does nothing -> all `|00>`
- `theta = pi/2`  -> qubit 0 in an even superposition, entangled -> roughly half `|00>`, half `|11>`
- `theta = pi`    -> `Rx(pi)` flips qubit 0 to |1>, CNOT flips qubit 1 -> all `|11>`

In [ ]:
shots = 2000
for v in [0.0, np.pi / 2, np.pi]:
    result = device.run(circ, inputs={"theta": float(v)}, shots=shots).result()
    counts = dict(result.measurement_counts)
    print(f"theta = {v:6.4f}  ->  {counts}")

Same compiled program, three different outcomes — driven entirely by `inputs`. This is exactly the reuse a variational optimizer needs: it proposes a new `theta` each step and resubmits without recompiling.

To see the parameter dependence as a smooth curve rather than sampled counts, attach an expectation observable and sweep `theta`. With `shots=0` the local simulator returns the exact expectation value, so `<Z_0>` traces out `cos(theta)`:

In [ ]:
circ_exp = Circuit().rx(0, FreeParameter("theta")).cnot(0, 1)
circ_exp.expectation(observable=Observable.Z(), target=0)

thetas = np.linspace(0.0, 2 * np.pi, 25)
z_exp = np.array([
    device.run(circ_exp, inputs={"theta": float(t)}, shots=0).result().values[0]
    for t in thetas
])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thetas, z_exp, "o-", color="#2b6cb0", label=r"$\langle Z_0 \rangle$ (local sim, exact)")
ax.plot(thetas, np.cos(thetas), "--", color="#888", label=r"$\cos(\theta)$")
ax.set_xlabel(r"$\theta$ (radians)")
ax.set_ylabel(r"$\langle Z_0 \rangle$")
ax.set_title("One FreeParameter circuit swept over theta")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("max deviation from cos(theta):", float(np.max(np.abs(z_exp - np.cos(thetas)))))

## 2. Modeling the compile-once speedup

Why does this matter for cost and wall-clock time? Because compilation is not free. On managed simulators and QPUs every submission of a freshly built circuit incurs a compile step before it can run.

Let `compile_cost` be the time to compile a program and `run_cost` the time to execute it once. For `N` iterations:

- **Recompile each iteration:** `N * (compile_cost + run_cost)`
- **Parametric (compile once):** `compile_cost + N * run_cost`

The model below uses representative seconds purely to illustrate the *shape* of the saving. It is a numpy timing **model**, not a measurement — real compile and run times depend on the device, circuit depth, and queue. The point is the structural difference, not the absolute numbers.

In [ ]:
# --- Illustrative timing MODEL (numpy only, no hardware, no AWS) ---
# These constants are stand-ins to show the scaling, not measured times.
compile_cost = 0.40  # seconds to compile a program (illustrative)
run_cost = 0.05      # seconds to execute one shot batch (illustrative)

N = np.arange(1, 101)
time_recompile = N * (compile_cost + run_cost)   # compile every iteration
time_parametric = compile_cost + N * run_cost    # compile once, reuse
speedup = time_recompile / time_parametric

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

ax1.plot(N, time_recompile, label="Recompile each iteration", color="#c53030")
ax1.plot(N, time_parametric, label="Parametric (compile once)", color="#2f855a")
ax1.fill_between(N, time_parametric, time_recompile, color="#2f855a", alpha=0.10)
ax1.set_xlabel("Iterations (N)")
ax1.set_ylabel("Total time (s, illustrative)")
ax1.set_title("Total loop time vs N (timing model)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(N, speedup, color="#2b6cb0")
ax2.axhline(1.0, color="#888", linestyle="--")
ax2.set_xlabel("Iterations (N)")
ax2.set_ylabel("Speedup (recompile / parametric)")
ax2.set_title("Speedup approaches compile/run ratio")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At N=50 (illustrative): recompile={time_recompile[49]:.2f}s, "
      f"parametric={time_parametric[49]:.2f}s, speedup={speedup[49]:.2f}x")
print(f"Asymptotic speedup ceiling = (compile + run)/run = "
      f"{(compile_cost + run_cost) / run_cost:.2f}x")

Two takeaways from the model:

1. The recompile line grows with slope `compile_cost + run_cost`; the parametric line grows with the much smaller slope `run_cost`. The gap (shaded) is the compile time you stop re-paying.
2. The speedup is bounded above by `(compile_cost + run_cost) / run_cost`. The more expensive compilation is relative to a single run, the more parametric compilation buys you — which is exactly the regime of deep variational circuits on real hardware.

## 3. Packaging it as a Hybrid Job

The local demo above is the whole idea. A Hybrid Job earns its keep when the same parametric loop runs longer, wants priority QPU access, and should be reproducible from a single artifact. The job's algorithm script builds the `FreeParameter` circuit once and submits it across the sweep — Braket reuses the compiled program inside the managed environment.

We define the entry point as a **string** so the notebook stays runnable with no credentials. Nothing here calls AWS; it is source code we will hand to the job.

In [ ]:
entry_point_source = '''
import numpy as np
import os
from braket.aws import AwsDevice
from braket.circuits import Circuit, FreeParameter
from braket.jobs import save_job_result
from braket.jobs.metrics import log_metric


def main():
    # Inside a running job, the device ARN is provided by the environment.
    device = AwsDevice(os.environ["AMZN_BRAKET_DEVICE_ARN"])

    # Build the parametric circuit ONCE; Braket compiles it once and reuses it.
    theta = FreeParameter("theta")
    circ = Circuit().rx(0, theta).cnot(0, 1)

    angles = np.linspace(0.0, np.pi, 50)
    p11_history = []
    for i, a in enumerate(angles):
        task = device.run(circ, inputs={"theta": float(a)}, shots=200)
        p11 = task.result().measurement_probabilities.get("11", 0.0)
        log_metric(metric_name="p11", value=p11, iteration_number=i)
        p11_history.append(p11)

    save_job_result({"p11": p11_history})


if __name__ == "__main__":
    main()
'''

# Sanity-check that the entry point is valid Python (compiles, does not run).
compile(entry_point_source, "algorithm_script.py", "exec")
print("Entry-point source is valid Python (", len(entry_point_source.splitlines()), "lines ).")

With the script in hand, submitting the job is a single `AwsQuantumJob.create(...)` call. It is gated behind `RUN_ON_AWS` and left `False` by default.

**COST NOTE:** Creating a Hybrid Job provisions a managed classical instance billed per second, and any QPU/managed-simulator tasks it submits are billed per task and per shot (see project cost rules in `CLAUDE.md`). Prototype on the `LocalSimulator` first; only flip `RUN_ON_AWS = True` when the algorithm is validated and you accept the charge.

In [ ]:
RUN_ON_AWS = False  # leave False; flip only when you accept job + per-task/shot cost

if RUN_ON_AWS:
    import os

    # Write the entry point to a real file the job can package.
    with open("algorithm_script.py", "w") as f:
        f.write(entry_point_source)

    # SV1 here keeps the demo cheap; swap for a QPU ARN only when validated.
    job = AwsQuantumJob.create(
        device="arn:aws:braket:::device/quantum-simulator/amazon/sv1",
        source_module="algorithm_script.py",
        entry_point="algorithm_script:main",
        job_name="parametric-compilation-demo",
        wait_until_complete=False,
    )
    print("Submitted:", job.arn)
    # job.state(), job.metrics(), job.result() also require AWS and are not
    # called at top level.
else:
    print("RUN_ON_AWS is False — no AWS calls made.")
    print("The local demo in sections 1-2 already proved the parametric reuse.")

## Exercises

Work through these on the `LocalSimulator` first. Keep `RUN_ON_AWS = False` unless you explicitly intend to incur job cost.

In [ ]:
# Exercise 1: Two free parameters.
# Build Circuit().rx(0, FreeParameter("a")).ry(1, FreeParameter("b")).cnot(0, 1)
# and run it on `device` with inputs={"a": ..., "b": ...} for a small grid of
# (a, b) values. Confirm one compiled circuit serves the whole grid.
# TODO: your code here


In [ ]:
# Exercise 2: Re-run the timing model with compile_cost = 2.0 and run_cost = 0.05
# (a deep circuit on a slow-to-compile backend). What does the asymptotic
# speedup ceiling become, and at what N do you reach 90% of it?
# TODO: your code here


In [ ]:
# Exercise 3: Turn the section-1 expectation sweep into a tiny optimizer.
# Using the exact <Z_0> from circ_exp (shots=0), do a coarse-to-fine search for
# the theta that minimizes <Z_0>. (Analytic answer: theta = pi.) Count how many
# circuit evaluations you used -- each is one inputs={...} run of the SAME circuit.
# TODO: your code here


## Summary

- `FreeParameter` lets you build a variational circuit **once** and drive it with `inputs={...}`; Braket compiles the program once and substitutes new values each run.
- We saw this for real on the `LocalSimulator`: one circuit object produced `|00>`, a `|00>`/`|11>` split, and `|11>` across three angles, and traced an exact `<Z_0> = cos(theta)` sweep.
- The compile-once saving is structural: total time goes from `N * (compile + run)` to `compile + N * run`, with a speedup ceiling of `(compile + run) / run`. The plotted numbers are an illustrative timing **model**, not measurements.
- The Hybrid Job wraps the same loop in a packaged entry point; `AwsQuantumJob.create(...)` stays behind `RUN_ON_AWS = False` with a cost note, so nothing in this notebook touches AWS.

**Next:** [`03-monitoring-metrics.ipynb`](03-monitoring-metrics.ipynb) — stream `log_metric` values from a running job and watch convergence live.